# Milestone 2: Data Quality dan Profiling

Lanjutan dari `01_data_profiling.ipynb` pada dataset **MedFact-Synth**. Notebook ini menerapkan profiling DuckDB `SUMMARIZE` dan transformasi Polars untuk menangani missing values, outlier, duplikat, serta validasi data dasar.

Pembagian peran:
- **DuckDB** untuk profiling dan validasi (`SUMMARIZE`, query SQL aturan kualitas).
- **Polars (lazy)** untuk transformasi dan penyimpanan hasil ke `data/processed/` lewat `sink_parquet` (streaming, tidak memuat semua data ke RAM).

In [1]:
from pathlib import Path

CWD = Path.cwd().resolve()
PROJECT_ROOT = CWD.parent if CWD.name == 'notebooks' else CWD
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
DATA_GLOB = (RAW_DIR / 'data' / '*.parquet').as_posix()

import duckdb
import polars as pl

OUTPUT_PATH = PROJECT_ROOT / 'data' / 'processed' / 'medfact_clean.parquet'

if not list((RAW_DIR / 'data').glob('*.parquet')):
    raise FileNotFoundError(
        f'Dataset tidak ditemukan di {RAW_DIR / "data"}. '
        'Pastikan Milestone 1 sudah menempatkan data di data/raw/.'
    )

lf = pl.scan_parquet(DATA_GLOB)
jumlah_baris = lf.select(pl.len()).collect(engine='streaming').item()
print(f'Ukuran awal: {jumlah_baris:,} baris x {len(lf.collect_schema())} kolom')
lf.head().collect(engine='streaming')

Ukuran awal: 1,497,981 baris x 10 kolom


idx,claim_pmid,claim_potential,claim,source_pmid,source,synthetic_label,system_prompt,user_prompt,assistant_prompt
i64,str,str,str,str,str,f64,str,str,str
0,"""30588127""","""support""","""An educational intervention us…","""30588127""","""An educational intervention ba…",2.0,"""You are a fact-checking expert…","""Article: An educational interv…","""<think>The article discusses a…"
1,"""30588127""","""support""","""An educational intervention us…","""24523424""","""A systematic review of barrier…",0.0,"""You are a fact-checking expert…","""Article: A systematic review o…","""<think>The article discusses b…"
2,"""30588127""","""support""","""An educational intervention us…","""30604386""","""The Effect of Educational Inte…",-1.0,"""You are a fact-checking expert…","""Article: The Effect of Educati…","""<think>The article discusses a…"
3,"""34346072""","""support""","""Older cancer survivors experie…","""34346072""","""Functional decline among older…",2.0,"""You are a fact-checking expert…","""Article: Functional decline am…","""<think>The article directly in…"
4,"""11255101""","""support""","""A strong humoral immune respon…","""11497076""","""Relationships between the pres…",2.0,"""You are a fact-checking expert…","""Article: Relationships between…","""<think>The article states that…"


## 1. Profiling dengan DuckDB `SUMMARIZE`

Dapatkan ringkasan statistik seluruh kolom (tipe data, null%, min/max, distinct count) dalam satu perintah.

In [2]:
connection = duckdb.connect()

profil = connection.execute(
    'SUMMARIZE SELECT * FROM read_parquet(?)', [DATA_GLOB]
).pl()
profil

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,str,str,str,str,str,i64,"decimal[9,2]"
"""idx""","""BIGINT""","""0""","""1497980""",1421176,"""748990.0""","""432430.01113301096""","""373535""","""749325""","""1123134""",1497981,0.00
"""claim_pmid""","""VARCHAR""","""1000020""","""999794""",350174,null,null,null,null,null,1497981,0.00
"""claim_potential""","""VARCHAR""","""refute""","""support""",2,null,null,null,null,null,1497981,0.00
"""claim""","""VARCHAR""","""""Backwards fluid-transporting …","""⁹⁹mTc-lactadherin is a sensiti…",608378,null,null,null,null,null,1497981,0.00
"""source_pmid""","""VARCHAR""","""100002""","""999923""",1273393,null,null,null,null,null,1497981,0.00
"""source""","""VARCHAR""",""""" Anoxygenic phototrophic bact…","""＜Editors' Choice＞ The relation…",1267661,null,null,null,null,null,1497981,0.00
"""synthetic_label""","""DOUBLE""","""-2.0""","""2.0""",5,"""0.1833027254684806""","""1.352350123096305""","""-0.8247391189741891""","""0.0""","""1.0""",1497981,0.00
"""system_prompt""","""VARCHAR""","""You are a fact-checking expert…","""You are a fact-checking expert…",1,null,null,null,null,null,1497981,0.00
"""user_prompt""","""VARCHAR""","""Article: "" Anoxygenic phototro…","""Article: ＜Editors' Choice＞ The…",1467976,null,null,null,null,null,1497981,0.00


## 2. Deteksi Missing Values

Lihat jumlah dan persentase null per kolom. kolom teks juga dicek untuk string kosong/spasi saja, karena itu juga bentuk data hilang yang tidak terdeteksi `null_count`.

**Alasan:** `null_count` hanya menghitung `null`, sedangkan data dari proses generasi sintetis bisa berisi string `''` atau spasi yang secara isi sama kosongnya. Mengecek keduanya mencegah baris kosong lolos ke analisis.

In [3]:
KOLOM_TEKS = ['claim_pmid', 'claim_potential', 'claim', 'source_pmid', 'source',
              'system_prompt', 'user_prompt', 'assistant_prompt']

null_summary = (
    lf.null_count().collect(engine='streaming')
    .transpose(include_header=True, header_name='kolom', column_names=['jumlah_null'])
    .with_columns((pl.col('jumlah_null') / jumlah_baris * 100).round(2).alias('persen_null'))
    .sort('jumlah_null', descending=True)
)
print(null_summary)

kosong = lf.select(
    [(pl.col(k).str.strip_chars() == '').sum().alias(k) for k in KOLOM_TEKS]
).collect(engine='streaming')
print()
print('String kosong per kolom:')
print(kosong)

shape: (10, 3)
┌──────────────────┬─────────────┬─────────────┐
│ kolom            ┆ jumlah_null ┆ persen_null │
│ ---              ┆ ---         ┆ ---         │
│ str              ┆ u32         ┆ f64         │
╞══════════════════╪═════════════╪═════════════╡
│ idx              ┆ 0           ┆ 0.0         │
│ claim_pmid       ┆ 0           ┆ 0.0         │
│ claim_potential  ┆ 0           ┆ 0.0         │
│ claim            ┆ 0           ┆ 0.0         │
│ source_pmid      ┆ 0           ┆ 0.0         │
│ source           ┆ 0           ┆ 0.0         │
│ synthetic_label  ┆ 0           ┆ 0.0         │
│ system_prompt    ┆ 0           ┆ 0.0         │
│ user_prompt      ┆ 0           ┆ 0.0         │
│ assistant_prompt ┆ 0           ┆ 0.0         │
└──────────────────┴─────────────┴─────────────┘



String kosong per kolom:
shape: (1, 8)
┌────────────┬─────────────┬───────┬─────────────┬────────┬─────────────┬─────────────┬────────────┐
│ claim_pmid ┆ claim_poten ┆ claim ┆ source_pmid ┆ source ┆ system_prom ┆ user_prompt ┆ assistant_ │
│ ---        ┆ tial        ┆ ---   ┆ ---         ┆ ---    ┆ pt          ┆ ---         ┆ prompt     │
│ u32        ┆ ---         ┆ u32   ┆ u32         ┆ u32    ┆ ---         ┆ u32         ┆ ---        │
│            ┆ u32         ┆       ┆             ┆        ┆ u32         ┆             ┆ u32        │
╞════════════╪═════════════╪═══════╪═════════════╪════════╪═════════════╪═════════════╪════════════╡
│ 0          ┆ 0           ┆ 0     ┆ 0           ┆ 0      ┆ 0           ┆ 0           ┆ 0          │
└────────────┴─────────────┴───────┴─────────────┴────────┴─────────────┴─────────────┴────────────┘


## 3. Penanganan Missing Values

Tidak ada kolom numerik yang perlu diimputasi: `claim`, `source`, dan label adalah isi utama tiap baris, sehingga nilai yang hilang tidak bisa diisi dengan median atau default tanpa memalsukan data. Strategi yang dipilih adalah **membuang baris** yang kolom kuncinya null atau string kosong. Pada dataset ini jumlahnya terdeteksi di langkah 2.

**Alasan:**
- Imputasi (median, modus, nilai default) hanya masuk akal untuk atribut numerik atau kategorik sederhana. `claim`, `source`, dan `assistant_prompt` adalah teks unik per baris, sehingga mengisinya sama dengan mengarang data.
- `synthetic_label` adalah target. Mengisinya dengan nilai tengah akan menanam label palsu yang merusak analisis dan pelatihan model.
- Baris yang dibuang sangat sedikit (lihat hitungan di bawah), jadi kehilangan informasinya dapat diabaikan.

In [4]:
KOLOM_KRITIS = ['idx', 'claim_pmid', 'claim_potential', 'claim', 'source_pmid',
                'source', 'synthetic_label', 'assistant_prompt']

kondisi_lengkap = pl.all_horizontal([pl.col(k).is_not_null() for k in KOLOM_KRITIS]) & pl.all_horizontal(
    [pl.col(k).str.strip_chars() != '' for k in KOLOM_KRITIS if k not in ('idx', 'synthetic_label')]
)

lf_missing_handled = lf.filter(kondisi_lengkap)

n_setelah_missing = lf_missing_handled.select(pl.len()).collect(engine='streaming').item()
print(f'Baris sebelum : {jumlah_baris:,}')
print(f'Baris sesudah : {n_setelah_missing:,}')
print(f'Terbuang      : {jumlah_baris - n_setelah_missing:,}')

Baris sebelum : 1,497,981
Baris sesudah : 1,497,981
Terbuang      : 0


## 4. Deteksi dan Penanganan Outlier

`synthetic_label` bersifat ordinal (skor kesesuaian -2 s.d. 2), jadi outlier dinilai dengan **batas domain**, bukan IQR. IQR dipakai untuk kolom numerik turunan, yaitu **panjang teks** `claim` dan `source`, untuk mengetahui seberapa ekstrem sebarannya.

**Alasan:** IQR mengandaikan data numerik kontinu. Pada label ordinal dengan hanya 5 nilai, IQR tidak bermakna dan bisa salah menandai label sah (misalnya -2) sebagai outlier. Rentang -2 s.d. 2 adalah aturan domain yang pasti dan lebih tepat.

In [5]:
LABEL_MIN, LABEL_MAX = -2.0, 2.0

print('Sebaran synthetic_label:')
print(lf_missing_handled.group_by('synthetic_label').agg(pl.len().alias('jumlah')).sort('synthetic_label').collect(engine='streaming'))

label_di_luar = lf_missing_handled.filter(
    ~pl.col('synthetic_label').is_between(LABEL_MIN, LABEL_MAX)
).select(pl.len()).collect(engine='streaming').item()
print(f'\nBaris dengan label di luar [{LABEL_MIN}, {LABEL_MAX}]: {label_di_luar:,}')

Sebaran synthetic_label:


shape: (5, 2)
┌─────────────────┬────────┐
│ synthetic_label ┆ jumlah │
│ ---             ┆ ---    │
│ f64             ┆ u32    │
╞═════════════════╪════════╡
│ -2.0            ┆ 266667 │
│ -1.0            ┆ 111819 │
│ 0.0             ┆ 545603 │
│ 1.0             ┆ 228047 │
│ 2.0             ┆ 345845 │
└─────────────────┴────────┘



Baris dengan label di luar [-2.0, 2.0]: 0


In [6]:
panjang = lf_missing_handled.select(
    pl.col('claim').str.len_chars().alias('panjang_claim'),
    pl.col('source').str.len_chars().alias('panjang_source'),
).collect(engine='streaming')

batas_iqr = {}
for kolom in panjang.columns:
    q1 = panjang[kolom].quantile(0.25)
    q3 = panjang[kolom].quantile(0.75)
    iqr = q3 - q1
    bawah, atas = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_outlier = panjang.filter((pl.col(kolom) < bawah) | (pl.col(kolom) > atas)).height
    batas_iqr[kolom] = (bawah, atas)
    print(f'{kolom}: Q1={q1:.0f}, Q3={q3:.0f}, IQR={iqr:.0f}, '
          f'batas=[{bawah:.0f}, {atas:.0f}], outlier={n_outlier:,} ({n_outlier / panjang.height * 100:.2f}%)')

print()
print(panjang.describe())

panjang_claim: Q1=112, Q3=169, IQR=57, batas=[26, 254], outlier=14,346 (0.96%)
panjang_source: Q1=1275, Q3=1908, IQR=633, batas=[326, 2858], outlier=29,198 (1.95%)

shape: (9, 3)
┌────────────┬───────────────┬────────────────┐
│ statistic  ┆ panjang_claim ┆ panjang_source │
│ ---        ┆ ---           ┆ ---            │
│ str        ┆ f64           ┆ f64            │
╞════════════╪═══════════════╪════════════════╡
│ count      ┆ 1.497981e6    ┆ 1.497981e6     │
│ null_count ┆ 0.0           ┆ 0.0            │
│ mean       ┆ 142.794385    ┆ 1609.395719    │
│ std        ┆ 41.171366     ┆ 529.651268     │
│ min        ┆ 28.0          ┆ 10.0           │
│ 25%        ┆ 112.0         ┆ 1275.0         │
│ 50%        ┆ 137.0         ┆ 1622.0         │
│ 75%        ┆ 169.0         ┆ 1908.0         │
│ max        ┆ 437.0         ┆ 14418.0        │
└────────────┴───────────────┴────────────────┘


Keputusan penanganan:
- **`synthetic_label`**: nilai di luar rentang domain dibuang (jika ada).
- **Panjang teks**: outlier IQR **tidak di-clip dan tidak dibuang**. Abstrak PubMed yang panjang adalah data sah, dan memotong teks akan merusak isi klaim/sumber. Outlier ini dicatat sebagai pertimbangan untuk tahap NLP (truncation).

**Alasan:** berbeda dengan windsorization pada angka, memotong teks mengubah maknanya sehingga label `synthetic_label` bisa tidak lagi cocok dengan isi klaim. Membuang barisnya juga berisiko membias data karena artikel panjang hilang secara sistematis.

## 5. Deduplikasi

Kunci bisnis dataset ini adalah `idx` (identifier baris). Dicek dua hal: duplikat kunci `idx` (jika `idx` unik maka tidak mungkin ada baris yang identik penuh), dan duplikat isi (`claim_pmid`, `source_pmid`, `claim`, `source` sama persis) yang dihitung lewat hash dengan engine streaming agar hemat memori. Duplikat isi hanya dilaporkan, tidak dibuang, karena baris dengan `idx` berbeda adalah sampel sintetis yang berbeda.

**Alasan:**
- Deduplikasi memakai kunci bisnis (`idx`), bukan seluruh baris, karena menghapus berdasarkan seluruh kolom bisa membuang baris sah yang kebetulan mirip.
- Hash dengan engine streaming dipakai agar tidak perlu memuat ±3 GB teks ke RAM hanya untuk menghitung duplikat.
- Kolom `idx` dikecualikan dari pengecekan duplikat isi karena nilainya selalu unik per baris.

In [7]:
KOLOM_KUNCI = 'idx'

n_unik_kunci = lf_missing_handled.select(pl.col(KOLOM_KUNCI).n_unique()).collect(engine='streaming').item()
KOLOM_ISI = ['claim_pmid', 'source_pmid', 'claim', 'source']
n_unik_isi = (
    lf_missing_handled.select(pl.struct(KOLOM_ISI).hash().n_unique())
    .collect(engine='streaming').item()
)

print(f'Total baris                    : {n_setelah_missing:,}')
print(f'Baris unik berdasarkan {KOLOM_KUNCI:<6}  : {n_unik_kunci:,}')
print(f'Duplikat kunci {KOLOM_KUNCI:<14}  : {n_setelah_missing - n_unik_kunci:,}')
print(f'Duplikat isi (4 kolom, hash)   : {n_setelah_missing - n_unik_isi:,} (dilaporkan saja)')

lf_dedup = lf_missing_handled
if n_setelah_missing > n_unik_kunci:
    lf_dedup = lf_dedup.unique(subset=[KOLOM_KUNCI], keep='first', maintain_order=False)
    print('Duplikat dibuang dengan unique(subset=idx).')
else:
    print('Tidak ada duplikat, tidak ada baris yang dibuang.')

Total baris                    : 1,497,981
Baris unik berdasarkan idx     : 1,497,981
Duplikat kunci idx             : 0
Duplikat isi (4 kolom, hash)   : 157 (dilaporkan saja)
Tidak ada duplikat, tidak ada baris yang dibuang.


## 6. Validasi Data Dasar

Aturan validasi sederhana yang ditulis sebagai query SQL DuckDB dan dihitung sebagai jumlah pelanggaran:

| Aturan | Keterangan |
|---|---|
| `label_valid` | `synthetic_label` berada pada -2 s.d. 2 |
| `potensi_valid` | `claim_potential` hanya `support` atau `refute` |
| `pmid_valid` | `claim_pmid` dan `source_pmid` hanya berisi digit |
| `format_think` | `assistant_prompt` diawali tag `<think>` (terdeteksi 1 sampel korup pada SUMMARIZE: `<th�think>`) |

**Alasan tiap aturan:**
- `label_valid`: skor di luar -2 s.d. 2 berarti data rusak, karena README mendefinisikan skala itu.
- `potensi_valid`: `claim_potential` adalah kategori tetap; nilai lain berarti salah ketik atau korup.
- `pmid_valid`: PMID PubMed selalu angka, sehingga huruf atau simbol menandakan ID rusak. Ini juga syarat agar `claim_pmid` bisa di-cast ke bilangan pada analisis proxy temporal (Milestone 3).
- `format_think`: format respons yang korup tidak dapat di-parse oleh tahap NLP berikutnya.
- Validasi dijalankan dengan SQL DuckDB agar terpisah dari kode pembersihan Polars, sehingga hasil keduanya bisa saling memeriksa.

In [8]:
def hitung_pelanggaran(sumber):
    return connection.execute(f"""
        SELECT
            COUNT(*) AS total_baris,
            SUM(CASE WHEN synthetic_label NOT BETWEEN {LABEL_MIN} AND {LABEL_MAX} THEN 1 ELSE 0 END) AS label_valid,
            SUM(CASE WHEN claim_potential NOT IN ('support', 'refute') THEN 1 ELSE 0 END) AS potensi_valid,
            SUM(CASE WHEN NOT regexp_matches(claim_pmid, '^[0-9]+$')
                       OR NOT regexp_matches(source_pmid, '^[0-9]+$') THEN 1 ELSE 0 END) AS pmid_valid,
            SUM(CASE WHEN NOT starts_with(assistant_prompt, '<think>') THEN 1 ELSE 0 END) AS format_think
        FROM {sumber}
    """).pl()


print('Pelanggaran pada data mentah:')
hitung_pelanggaran(f"read_parquet('{DATA_GLOB}')")

Pelanggaran pada data mentah:


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

total_baris,label_valid,potensi_valid,pmid_valid,format_think
i64,"decimal[38,0]","decimal[38,0]","decimal[38,0]","decimal[38,0]"
1497981,0,0,0,29


## 7. Simpan Data Bersih

Semua aturan di atas diterapkan sebagai filter, lalu hasilnya ditulis ke `data/processed/` lewat `sink_parquet` agar dapat dipakai pada Milestone 3 tanpa mengulang proses cleaning.

**Alasan:**
- `sink_parquet` menulis hasil secara streaming sehingga pembersihan data ±3 GB tidak membutuhkan RAM sebesar datanya.
- Hasil disimpan sebagai file terpisah di `data/processed/`, sedangkan `data/raw/` tidak diubah, sehingga proses dapat direproduksi dari data asli.
- Validasi diulang pada data bersih sebagai bukti bahwa seluruh pelanggaran sudah nol.

In [9]:
lf_bersih = (
    lf_dedup
    .filter(pl.col('synthetic_label').is_between(LABEL_MIN, LABEL_MAX))
    .filter(pl.col('claim_potential').is_in(['support', 'refute']))
    .filter(pl.col('claim_pmid').str.contains(r'^\d+$') & pl.col('source_pmid').str.contains(r'^\d+$'))
    .filter(pl.col('assistant_prompt').str.starts_with('<think>'))
)

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
lf_bersih.sink_parquet(OUTPUT_PATH)

hasil = pl.scan_parquet(OUTPUT_PATH)
n_bersih = hasil.select(pl.len()).collect(engine='streaming').item()

print(f'Baris sebelum cleaning: {jumlah_baris:,}')
print(f'Baris sesudah cleaning: {n_bersih:,}')
print(f'Baris terbuang        : {jumlah_baris - n_bersih:,} ({(jumlah_baris - n_bersih) / jumlah_baris * 100:.4f}%)')
print(f'Data bersih tersimpan di: {OUTPUT_PATH}')
print()
print('Pelanggaran pada data bersih:')
hitung_pelanggaran(f"read_parquet('{OUTPUT_PATH.as_posix()}')")

Baris sebelum cleaning: 1,497,981
Baris sesudah cleaning: 1,497,952
Baris terbuang        : 29 (0.0019%)
Data bersih tersimpan di: /home/jovyan/work/data/processed/medfact_clean.parquet

Pelanggaran pada data bersih:


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

total_baris,label_valid,potensi_valid,pmid_valid,format_think
i64,"decimal[38,0]","decimal[38,0]","decimal[38,0]","decimal[38,0]"
1497952,0,0,0,0


## Catatan Milestone 2

1. Dimensi kualitas yang paling bermasalah validitas format, bukan kelengkapan. Pemeriksaan awal pada data mentah: null 0, string kosong 0, `idx` duplikat 0, `synthetic_label` di luar -2 s.d. 2 sebanyak 0, `claim_potential` di luar `support`/`refute` 0, PMID non-digit 0, dan **29 baris** dengan `assistant_prompt` tidak diawali `<think>`.
2. Missing values tidak ada yang perlu diimputasi. Strategi yang dipilih adalah membuang baris jika kolom kunci null atau kosong (tidak ada yang terkena pada dataset ini). Imputasi median/modus ditolak karena `claim` dan `source` adalah isi utama baris dan tidak bisa diisi tanpa memalsukan data.
3. Outlier `synthetic_label` dinilai dengan batas domain (-2 s.d. 2) karena ordinal. Panjang teks `claim` dan `source` dianalisis dengan IQR; outlier panjang tidak di-clip dan tidak dibuang karena abstrak panjang adalah data sah. Ini dicatat sebagai bahan keputusan truncation pada tahap NLP.
4. Koreksi terhadap rencana awal filter label `[-1.0, 0.0, 2.0]` pada draf pertama salah karena akan membuang label -2 dan 1 (±494 ribu baris). Nilai valid yang benar adalah seluruh rentang -2 s.d. 2.
5. Duplikat kunci `idx` unik, sehingga tidak ada duplikat. Duplikat isi (`claim_pmid`, `source_pmid`, `claim`, `source`) hanya dilaporkan, tidak dibuang.
6. Jumlah baris berubah dari 1.497.981 menjadi sekitar 1.497.952 (selisih 29 baris, ±0,002%). Angka pasti tercetak pada cell terakhir bagian 7.
7. Validasi empat aturan SQL DuckDB (`label_valid`, `potensi_valid`, `pmid_valid`, `format_think`) dihitung pada data mentah dan diulang pada data bersih; target semua pelanggaran = 0.